# FabricNet — Fashion MNIST Classifier

**What you will learn in this notebook:**
1. Dense Neural Networks — what they are and how they work
2. Activation Functions — ReLU, Softmax, and when to use each
3. Overfitting — how to spot it in training curves
4. Regularisation (L2) — penalise large weights
5. Dropout — randomly silence neurons
6. Batch Normalisation — stabilise layer inputs
7. How to compare models and measure improvement

**Dataset:** Fashion MNIST — 70,000 grayscale images of 10 clothing categories  
**Framework:** TensorFlow 2.x / Keras

---

## The 10 classes
```
0: T-shirt/top    1: Trouser    2: Pullover    3: Dress      4: Coat
5: Sandal         6: Shirt      7: Sneaker     8: Bag        9: Ankle boot
```

## The plan
We train TWO models:
- **Baseline** — no regularisation, intentionally shows overfitting
- **Regularised** — L2 + Dropout + BatchNorm, shows the fix

Seeing the problem and the solution side-by-side is the best way to understand both.

## Cell 1 — Imports

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print('TensorFlow:', tf.__version__)
np.random.seed(42)
tf.random.set_seed(42)

CLASS_NAMES = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress',    'Coat',
    'Sandal',      'Shirt',   'Sneaker',  'Bag',       'Ankle boot',
]

## Cell 2 — Load & Explore the Data

Fashion MNIST is a drop-in replacement for handwritten digits (MNIST).
It's harder — clothing categories have more visual variation than digits.

In [ ]:
(x_train_raw, y_train_raw), (x_test_raw, y_test_raw) = keras.datasets.fashion_mnist.load_data()

print('Raw shapes:')
print(f'  x_train: {x_train_raw.shape}  dtype: {x_train_raw.dtype}  range: [{x_train_raw.min()}, {x_train_raw.max()}]')
print(f'  y_train: {y_train_raw.shape}  unique labels: {np.unique(y_train_raw)}')

# Visualise one sample per class
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('One sample per class', fontsize=12)
for label, ax in enumerate(axes.flat):
    idx = np.where(y_train_raw == label)[0][0]
    ax.imshow(x_train_raw[idx], cmap='gray')
    ax.set_title(CLASS_NAMES[label], fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Cell 3 — Preprocessing

Three steps before the data enters the network:

**Step 1 — Normalise:** pixels [0,255] → [0.0, 1.0]
- Weight updates are proportional to input magnitude
- Large inputs → large gradients → unstable training
- After normalising, all inputs are in the same scale

**Step 2 — Flatten:** (28, 28) → (784,)
- Dense layers need 1-D vectors, not 2-D grids

**Step 3 — One-hot encode:** 3 → [0,0,0,1,0,0,0,0,0,0]
- Required by categorical cross-entropy loss
- Categorical CE expects a probability distribution over classes

In [ ]:
# Step 1: Normalise
x_train = x_train_raw.astype('float32') / 255.0
x_test  = x_test_raw.astype('float32')  / 255.0

# Step 2: Flatten
x_train = x_train.reshape(-1, 784)
x_test  = x_test.reshape(-1, 784)

# Save integer labels for evaluation later
y_test_int = y_test_raw.copy()

# Step 3: One-hot encode
y_train = keras.utils.to_categorical(y_train_raw, 10)
y_test  = keras.utils.to_categorical(y_test_raw,  10)

print('After preprocessing:')
print(f'  x_train: {x_train.shape}  range: [{x_train.min():.1f}, {x_train.max():.1f}]')
print(f'  y_train: {y_train.shape}  (one-hot)')
print(f'  Example label: {y_train_raw[0]} → {y_train[0]}')

## Cell 4 — Activation Functions (visual reference)

Before building the network, let's visualise what the activations actually do.

In [ ]:
x = np.linspace(-4, 4, 300)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Activation functions', fontsize=12)

# ReLU
axes[0].plot(x, np.maximum(0, x), lw=2.5, color='steelblue')
axes[0].axhline(0, color='gray', lw=0.5); axes[0].axvline(0, color='gray', lw=0.5)
axes[0].set_title('ReLU: max(0, x)\nUsed in hidden layers')
axes[0].set_xlabel('Input'); axes[0].set_ylabel('Output')
axes[0].fill_between(x, 0, np.maximum(0, x), alpha=0.1)
axes[0].grid(alpha=0.3)
axes[0].annotate('dead zone\n(gradient=0)', xy=(-2, 0.1), fontsize=9, color='red')
axes[0].annotate('alive zone\n(gradient=1)', xy=(1, 2), fontsize=9, color='green')

# Sigmoid
axes[1].plot(x, 1/(1+np.exp(-x)), lw=2.5, color='coral')
axes[1].axhline(0.5, color='gray', lw=0.5, ls='--')
axes[1].set_title('Sigmoid: 1/(1+e⁻ˣ)\nSquashes to (0,1)')
axes[1].set_xlabel('Input')
axes[1].grid(alpha=0.3)
axes[1].annotate('saturates here →\ngradient ≈ 0', xy=(2.5, 0.9), fontsize=9, color='red')

# Softmax demo
logits  = np.array([-1.0, 0.5, 2.0, 0.8, -0.3])
softmax = np.exp(logits) / np.exp(logits).sum()
axes[2].bar(range(5), logits,  alpha=0.5, label='Raw logits')
axes[2].bar(range(5), softmax, alpha=0.8, color='teal', label='Softmax probs')
axes[2].set_xticks(range(5))
axes[2].set_xticklabels([f'C{i}' for i in range(5)])
axes[2].set_title(f'Softmax: converts logits → probs\nSum = {softmax.sum():.3f}')
axes[2].legend(); axes[2].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print('Key takeaway:')
print('  ReLU hidden layers → gradients flow freely for positive inputs')
print('  Softmax output    → 10 probabilities that sum to exactly 1.0')

## Cell 5 — Dense Neural Network Concept

Let's manually compute what one Dense layer does to understand the math.

In [ ]:
# Manual Dense layer computation
np.random.seed(0)
n_inputs  = 4
n_outputs = 3

x_example = np.array([0.5, -0.2, 0.8, 0.1])          # one sample, 4 features
W = np.random.randn(n_outputs, n_inputs) * 0.1         # weight matrix (3, 4)
b = np.zeros(n_outputs)                                 # bias vector (3,)

# Linear step: Z = W · x + b
Z = W @ x_example + b

# Non-linear step: A = ReLU(Z)
A = np.maximum(0, Z)

print('Dense layer computation:')
print(f'  Input x   : {x_example}  shape: {x_example.shape}')
print(f'  Weight W  : shape {W.shape}  ({n_outputs} neurons × {n_inputs} inputs)')
print(f'  Z = W·x+b : {np.round(Z, 4)}  (linear)')
print(f'  A = ReLU  : {np.round(A, 4)}  (non-linear)')
print()
print(f'Total weights in this layer: {n_outputs} × {n_inputs} + {n_outputs} (biases) = {n_outputs*n_inputs + n_outputs}')
print()
print(f'For our model: Dense(512) → {784*512 + 512:,} parameters in layer 1 alone!')

## Cell 6 — Build the BASELINE Model

No regularisation. We build this to **intentionally show overfitting**.

In [ ]:
def build_baseline():
    model = keras.Sequential([
        keras.Input(shape=(784,)),
        layers.Dense(512, activation='relu',    name='hidden_1'),
        layers.Dense(256, activation='relu',    name='hidden_2'),
        layers.Dense(10,  activation='softmax', name='output'),
    ], name='baseline')
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model

baseline = build_baseline()
baseline.summary()

print('\nParameter count breakdown:')
for layer in baseline.layers:
    if layer.count_params() > 0:
        print(f'  {layer.name:12s}: {layer.count_params():>10,} params')

## Cell 7 — Train the Baseline

Watch the training output carefully. After the first few epochs:
- `accuracy` (train) will keep climbing
- `val_accuracy` will plateau or diverge

That gap = **overfitting**.

In [ ]:
history_base = baseline.fit(
    x_train, y_train,
    epochs=30,
    batch_size=256,
    validation_data=(x_test, y_test),
    verbose=1,
)

train_acc = max(history_base.history['accuracy'])
val_acc   = max(history_base.history['val_accuracy'])
gap       = train_acc - val_acc
print(f'\nBest train accuracy : {train_acc*100:.2f}%')
print(f'Best val  accuracy  : {val_acc*100:.2f}%')
print(f'Overfitting gap     : {gap*100:.2f}%  ← this is the problem')

## Cell 8 — Visualise Overfitting

This is the most important plot so far. Learn to read this chart.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Baseline model — OVERFITTING', fontsize=12, color='red')

ep = range(1, len(history_base.history['loss']) + 1)

ax1.plot(ep, history_base.history['accuracy'],     label='Train', lw=2)
ax1.plot(ep, history_base.history['val_accuracy'], label='Val',   lw=2, ls='--')
ax1.fill_between(ep,
                 history_base.history['val_accuracy'],
                 history_base.history['accuracy'],
                 alpha=0.15, color='red', label='Overfitting gap')
ax1.set_title('Accuracy — growing gap = overfitting')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(ep, history_base.history['loss'],     label='Train', lw=2)
ax2.plot(ep, history_base.history['val_loss'], label='Val',   lw=2, ls='--')
ax2.set_title('Loss — val loss rises while train falls')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print('What you see:')
print('  Train loss falls continuously')
print('  Val loss starts rising after a few epochs → model memorising train data')
print('  Growing red shaded area = increasing overfitting')

## Cell 9 — Build the REGULARISED Model

Three defences against overfitting:

**L2 Regularisation:**
```python
kernel_regularizer=regularizers.L2(1e-4)
```
Adds `λ × Σ w²` to the loss. Pushes weights toward zero.
Large λ = stronger shrinkage. Start with 1e-4.

**Batch Normalisation:**
```python
layers.BatchNormalization()
```
Normalises each mini-batch to zero mean, unit variance.
Place after Dense, before activation.

**Dropout:**
```python
layers.Dropout(0.4)   # randomly zero 40% of neurons
```
Only active during training. At inference, all neurons are on.

In [ ]:
def build_regularised(l2=1e-4, drop1=0.4, drop2=0.3):
    reg = regularizers.L2(l2)
    model = keras.Sequential([
        keras.Input(shape=(784,)),

        # Layer 1: Dense → BN → ReLU → Dropout
        layers.Dense(512, kernel_regularizer=reg, name='hidden_1'),
        layers.BatchNormalization(name='bn_1'),
        layers.Activation('relu', name='relu_1'),
        layers.Dropout(drop1, name='dropout_1'),

        # Layer 2: Dense → BN → ReLU → Dropout
        layers.Dense(256, kernel_regularizer=reg, name='hidden_2'),
        layers.BatchNormalization(name='bn_2'),
        layers.Activation('relu', name='relu_2'),
        layers.Dropout(drop2, name='dropout_2'),

        # Output
        layers.Dense(10, activation='softmax', name='output'),
    ], name='regularised')

    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model

regularised = build_regularised()
regularised.summary()

## Cell 10 — Train the Regularised Model

With proper callbacks: EarlyStopping + ReduceLROnPlateau.

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=8,
        restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.3,
        patience=4, min_lr=1e-6, verbose=1
    ),
]

history_reg = regularised.fit(
    x_train, y_train,
    epochs=50,
    batch_size=256,
    validation_data=(x_test, y_test),
    callbacks=callbacks,
    verbose=1,
)

train_acc = max(history_reg.history['accuracy'])
val_acc   = max(history_reg.history['val_accuracy'])
gap       = train_acc - val_acc
print(f'\nBest train accuracy : {train_acc*100:.2f}%')
print(f'Best val  accuracy  : {val_acc*100:.2f}%')
print(f'Overfitting gap     : {gap*100:.2f}%  ← should be much smaller now')

## Cell 11 — THE KEY COMPARISON

Baseline vs Regularised — side by side.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Overfitting vs Regularisation\n'
             'Left: baseline   Right: regularised', fontsize=13)

for col, (title, h) in enumerate([('Baseline', history_base), ('Regularised', history_reg)]):
    ep = range(1, len(h.history['loss']) + 1)
    gap_text = f"Gap: {(max(h.history['accuracy'])-max(h.history['val_accuracy']))*100:.1f}%"
    color = 'red' if col == 0 else 'green'

    axes[0,col].plot(ep, h.history['accuracy'],     label='Train', lw=2)
    axes[0,col].plot(ep, h.history['val_accuracy'], label='Val',   lw=2, ls='--')
    axes[0,col].set_title(f'{title} — Accuracy')
    axes[0,col].set_xlabel('Epoch'); axes[0,col].set_ylabel('Accuracy')
    axes[0,col].legend(); axes[0,col].grid(alpha=0.3)
    axes[0,col].set_ylim(0.7, 1.0)
    axes[0,col].text(0.97, 0.05, gap_text, transform=axes[0,col].transAxes,
                     ha='right', fontsize=11, color=color, fontweight='bold')

    axes[1,col].plot(ep, h.history['loss'],     label='Train', lw=2)
    axes[1,col].plot(ep, h.history['val_loss'], label='Val',   lw=2, ls='--')
    axes[1,col].set_title(f'{title} — Loss')
    axes[1,col].set_xlabel('Epoch'); axes[1,col].set_ylabel('Loss')
    axes[1,col].legend(); axes[1,col].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('overfitting_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('Observation:')
print('  Baseline    → large gap between train and val accuracy')
print('  Regularised → train and val accuracy track closely together')

## Cell 12 — Evaluate Both Models

In [ ]:
_, acc_base = baseline.evaluate(x_test, y_test, verbose=0)
_, acc_reg  = regularised.evaluate(x_test, y_test, verbose=0)

print(f'Baseline    test accuracy : {acc_base*100:.2f}%')
print(f'Regularised test accuracy : {acc_reg*100:.2f}%')
print(f'Improvement               : {(acc_reg-acc_base)*100:+.2f}%')
print()

# Per-class report
y_pred = np.argmax(regularised.predict(x_test, verbose=0), axis=1)
print('Per-class performance (regularised model):')
print(classification_report(y_test_int, y_pred, target_names=CLASS_NAMES, digits=3))

## Cell 13 — Confusion Matrix

Shows where the model gets confused.  
Look for the hardest classes — Shirt vs T-shirt is a common confusion.

In [ ]:
cm      = confusion_matrix(y_test_int, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.4, ax=ax)
ax.set_title('Confusion Matrix — Regularised Model\n(normalised per true class)', fontsize=12)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('Hardest classes (lowest diagonal values):')
diag = np.diag(cm_norm)
for idx in np.argsort(diag)[:3]:
    print(f'  {CLASS_NAMES[idx]:15s}: {diag[idx]*100:.1f}% correct')

## Cell 14 — Sample Predictions

In [ ]:
n = 20
idx   = np.random.choice(len(x_test), n, replace=False)
imgs  = x_test[idx]
true  = y_test_int[idx]
pred  = np.argmax(regularised.predict(imgs, verbose=0), axis=1)
conf  = regularised.predict(imgs, verbose=0).max(axis=1)

fig, axes = plt.subplots(4, 5, figsize=(14, 11))
fig.suptitle('Sample predictions (green=correct, red=wrong)', fontsize=11)
for i, ax in enumerate(axes.flat):
    ax.imshow(imgs[i].reshape(28, 28), cmap='gray')
    ok    = pred[i] == true[i]
    color = 'green' if ok else 'red'
    ax.set_title(f'P: {CLASS_NAMES[pred[i]]}\nT: {CLASS_NAMES[true[i]]}\n{conf[i]*100:.0f}%',
                 fontsize=6.5, color=color)
    ax.axis('off')
plt.tight_layout()
plt.savefig('sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell 15 — Weight Distributions

Visualise the effect of L2 regularisation on weight values.
Regularised weights should be more concentrated around zero.

In [ ]:
def get_all_weights(model):
    return np.concatenate([
        layer.get_weights()[0].ravel()
        for layer in model.layers if layer.get_weights()
    ])

w_base = get_all_weights(baseline)
w_reg  = get_all_weights(regularised)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
fig.suptitle('Weight distributions — L2 regularisation keeps weights small', fontsize=12)

axes[0].hist(w_base, bins=80, color='steelblue', alpha=0.8)
axes[0].set_title(f'Baseline  |  std={w_base.std():.4f}')
axes[0].set_xlabel('Weight value'); axes[0].set_ylabel('Count'); axes[0].grid(alpha=0.3)

axes[1].hist(w_reg, bins=80, color='coral', alpha=0.8)
axes[1].set_title(f'Regularised (L2)  |  std={w_reg.std():.4f}')
axes[1].set_xlabel('Weight value'); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('weight_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Baseline    weight std: {w_base.std():.5f}')
print(f'Regularised weight std: {w_reg.std():.5f}')
print('Smaller std = weights more concentrated near zero = less overfitting capacity')

## Cell 16 — Dropout: Training vs Inference

Dropout behaves differently during training and inference.  
Let's verify this directly.

In [ ]:
sample = x_test[:1]

# During training mode (training=True) — Dropout is active
# Run the same sample multiple times — predictions will differ!
train_preds = []
for _ in range(10):
    pred = regularised(sample, training=True).numpy()[0]
    train_preds.append(pred)
train_preds = np.array(train_preds)

# During inference mode (training=False) — Dropout is off
# All 10 runs produce the same result
infer_preds = []
for _ in range(10):
    pred = regularised(sample, training=False).numpy()[0]
    infer_preds.append(pred)
infer_preds = np.array(infer_preds)

print('Dropout behaviour:')
print(f'  Training mode  — prediction std across 10 runs: {train_preds.std(axis=0).mean():.4f}  (varies)')
print(f'  Inference mode — prediction std across 10 runs: {infer_preds.std(axis=0).mean():.6f} (identical)')
print()
print('This confirms Dropout only activates during training, not at inference time.')

## Summary

| Concept | What it does | Where used |
|---|---|---|
| Dense layer | Every input connects to every output | All hidden layers |
| ReLU | `max(0, x)` — zero negatives | Hidden layers |
| Softmax | Converts logits to probabilities summing to 1 | Output layer |
| Overfitting | Train acc >> val acc — model memorising | The problem |
| L2 reg | Penalise large weights via `λΣw²` | Every Dense layer |
| Batch Norm | Normalise mini-batch activations | After Dense |
| Dropout | Randomly silence neurons during training | After activation |
| EarlyStopping | Halt when val_loss stops improving | Callback |
| ReduceLROnPlateau | Reduce learning rate on plateau | Callback |

## What's Next

Dense networks treat each pixel independently — they have no sense of spatial structure.  
A **Convolutional Neural Network (CNN)** processes the image as a 2-D grid, detecting
edges and textures in local patches before combining them into global features.
CNNs typically reach 93–95% on Fashion MNIST vs 89–91% for dense networks.